# Weighted Stage B Training

This notebook trains only Stage B with inverse-frequency weighted cross-entropy. It preserves the baseline checkpoint in `ml/outputs_two_stage/stage_b/best_model` and writes the weighted checkpoint to `ml/outputs_two_stage/stage_b_weighted/best_model`.

In [ ]:
# Colab setup: run from a fresh runtime.
!pip install -q -r requirements.txt
!pip install -q -r backend/requirements.txt
!pip install -q -r ml/requirements.txt

In [ ]:
from pathlib import Path
import sys
import torch

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'ml' / 'train_stage_b_weighted.py').exists():
    raise FileNotFoundError('Run this notebook from the repository root after cloning the project.')
sys.path.insert(0, str(PROJECT_ROOT / 'ml'))

print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Enable a CUDA GPU runtime before training.')
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
from config import STAGE_B_ID2LABEL, STAGE_B_NUM_LABELS, TRAIN_CSV
from train_stage_b_weighted import _stage_b_rows, class_weights

train_rows = _stage_b_rows(TRAIN_CSV)
weights, counts = class_weights(train_rows)
print('model_index -> class_name -> count -> weight')
for index in range(STAGE_B_NUM_LABELS):
    print(index, '->', STAGE_B_ID2LABEL[index], '->', counts[index], '->', round(weights[index].item(), 4))

In [ ]:
from train_stage_b_weighted import train

# Trains only Stage B and saves a separate checkpoint.
train()

In [ ]:
import json
from pathlib import Path

summary_path = Path('ml/outputs_two_stage/stage_b_weighted/weighted_evaluation.json')
summary = json.loads(summary_path.read_text(encoding='utf-8'))
print(json.dumps(summary, indent=2))